# 2. De-identification

Detect protected health information (PHI), transform it, and get an auditable
report - then measure how well it worked.

> Every note, patient, number and identifier in this notebook is **fictitious**. It runs offline, downloads no model, and uses no real patient data.

> **What the default recognizer does not do.** It finds *structured* identifiers
> (SSN, MRN, phone, dates, e-mail, ...). It does **not** find names or street
> addresses; that needs the opt-in NER recognizer. This notebook shows the gap
> rather than hiding it.

In [1]:
import os

# Keep OpenBTK's routine debug lines out of this notebook's output.
os.environ.setdefault("OPENBTK_LOG_LEVEL", "warning")

'warning'

In [2]:
from openbtk.deid import DeidEngine, DeidMode

TEXT = "Seen 03/14/2024 by Dr Jane Roe. Call (555) 010-2345 to confirm."

engine = DeidEngine(mode=DeidMode.REDACT, recognizers=["rule"])
result = engine.deidentify(TEXT, patient_id="hashed-patient-id")
result.text

'Seen [REDACTED] by Dr Jane Roe. Call [REDACTED] to confirm.'

The date and phone number are gone; **`Jane Roe` is still there.** That is the
rule recognizer's documented limit, not a bug in this example.

## The report

It holds categories, counts and offsets - never the values it found - so it is
safe to archive or attach to a manifest.

In [3]:
report = result.report
print({c.value: n for c, n in report.entity_counts.items()})
print("risk:", report.residual_risk.level, "-", report.residual_risk.rationale)
assert "010-2345" not in report.model_dump_json()

{'date': 1, 'phone_number': 1}
risk: low - 2 detection(s) survived filtering; lowest post-merge confidence was 0.95.


## Five transform modes

The same text, five ways of replacing what was found.

In [4]:
for mode in DeidMode:
    out = DeidEngine(mode=mode, recognizers=["rule"]).deidentify(
        TEXT, patient_id="hashed-patient-id"
    )
    print(f"{mode.value:11} {out.text}")

redact      Seen [REDACTED] by Dr Jane Roe. Call [REDACTED] to confirm.
surrogate   Seen [DATE_1] by Dr Jane Roe. Call [PHONE_NUMBER_1] to confirm.
hash        Seen [DATE_HASH_2962da93] by Dr Jane Roe. Call [PHONE_NUMBER_HASH_6f354e6c] to confirm.
tag         Seen [DATE] by Dr Jane Roe. Call [PHONE_NUMBER] to confirm.
date_shift  Seen 2023-07-07 by Dr Jane Roe. Call [REDACTED] to confirm.


`SURROGATE` swaps in realistic fake values consistently per patient, and
`DATE_SHIFT` moves every date by the same per-patient offset so the *intervals*
between events survive.

## Adding names: the NER recognizer

Names need the spaCy-based recognizer (`pip install "openbtk[text]"` and
`python -m spacy download en_core_web_sm`). This cell uses it if it is installed
and says so if not.

In [5]:
import importlib.util

if importlib.util.find_spec("spacy") and importlib.util.find_spec("en_core_web_sm"):
    ner = DeidEngine(mode=DeidMode.REDACT, recognizers=["rule", "ner"])
    print(ner.deidentify(TEXT, patient_id="hashed-patient-id").text)
else:
    print(
        "NER is not installed here - install openbtk[text] and the spaCy model to enable it."
    )

Seen [REDACTED] by Dr [REDACTED]. Call [REDACTED] to confirm.


## Measure it on labelled data

Detection quality depends on your notes. Give the scorer spans a human has
labelled and it reports precision, recall and F1 per Safe Harbor category, plus a
category-agnostic "any PHI" view. Matching is *relaxed* (any overlap counts), which
is more forgiving than boundary-exact scoring - do not compare these numbers with
papers that use a stricter rule.

In [6]:
from IPython.display import Markdown, display

from openbtk.deid.labelled import LabelledDocument, LabelledSpan
from openbtk.deid.schemas import PHICategory
from openbtk.eval.deid import evaluate_deid, format_markdown


def labelled(category, substring):
    start = TEXT.index(substring)
    return LabelledSpan(category=category, start=start, end=start + len(substring))


doc = LabelledDocument(
    document_id="d1",
    text=TEXT,
    spans=[
        labelled(PHICategory.DATE, "03/14/2024"),
        labelled(PHICategory.NAME, "Jane Roe"),
        labelled(PHICategory.PHONE_NUMBER, "(555) 010-2345"),
    ],
)

scores = evaluate_deid(engine, [doc])
display(Markdown(format_markdown(scores, title="rule recognizer, one labelled note")))

### rule recognizer, one labelled note

1 documents; matching: relaxed span overlap (any overlapping detection counts).

| Category | TP | FP | FN | Precision | Recall | F1 |
|---|---:|---:|---:|---:|---:|---:|
| date | 1 | 0 | 0 | 1.000 | 1.000 | 1.000 |
| name | 0 | 0 | 1 | 1.000 | 0.000 | 0.000 |
| phone_number | 1 | 0 | 0 | 1.000 | 1.000 | 1.000 |
| **overall (category-aware)** | 2 | 0 | 1 | 1.000 | 0.667 | 0.800 |
| **binary (any PHI)** | 2 | 0 | 1 | 1.000 | 0.667 | 0.800 |


The name row is a false negative: exactly the gap the warning at the top
described. On the project's synthetic test corpus the published numbers, and what
they do and do not show, are on the
[Benchmarks](https://openbtk.org/openbtk-core/dev/benchmarks/) page. **No
i2b2/n2c2 result is published** - that corpus needs a data use agreement.